# Topic and Age Distribution Analysis

This notebook analyzes the distribution of topics across different age groups from Amazon book data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10


## 1. Load All Amazon Data


In [ ]:
# Load all Amazon CSV files from the amazon_data folder
amazon_data_folder = Path("../data/amazon_data")

all_dataframes = []

if amazon_data_folder.exists():
    csv_files = list(amazon_data_folder.glob("*.csv"))
    print(f"Found {len(csv_files)} Amazon data CSV files")
    
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
            if not df.empty:
                df['source_file'] = csv_file.stem
                all_dataframes.append(df)
        except Exception as e:
            print(f"Error loading {csv_file.name}: {e}")
    
    if all_dataframes:
        combined_df = pd.concat(all_dataframes, ignore_index=True)
        print(f"\nCombined dataset shape: {combined_df.shape}")
        print(f"Columns: {combined_df.columns.tolist()}")
    else:
        print("No data found in CSV files")
        combined_df = pd.DataFrame()
else:
    print(f"Amazon data folder not found: {amazon_data_folder}")
    combined_df = pd.DataFrame()


## 2. Extract Topics from Categories


In [ ]:
def extract_topics_from_categories(categories_str):
    """Extract individual topics from Amazon categories"""
    if pd.isna(categories_str) or categories_str == 'N/A':
        return []
    
    # Split by ' > ' to get hierarchical categories
    categories = str(categories_str).split(' > ')
    
    # Filter out generic categories like 'Books', 'Kindle Store', etc.
    generic_categories = {'books', 'kindle store', 'kindle', 'ebooks', 'audible', 'prime reading'}
    
    topics = []
    for cat in categories:
        cat_clean = cat.strip().lower()
        if cat_clean and cat_clean not in generic_categories:
            topics.append(cat.strip())
    
    return topics

# Extract topics
if not combined_df.empty:
    combined_df['topics'] = combined_df['Categories (Amazon)'].apply(extract_topics_from_categories)
    
    # Flatten topics for analysis
    all_topics = []
    for topics_list in combined_df['topics']:
        all_topics.extend(topics_list)
    
    topic_counts = Counter(all_topics)
    print(f"\nTotal unique topics: {len(topic_counts)}")
    print(f"\nTop 20 most common topics:")
    for topic, count in topic_counts.most_common(20):
        print(f"  {topic}: {count}")


## 3. Parse Age Ranges and Expand Them


In [ ]:
def parse_age_range(age_str):
    """Parse age range from various formats and return list of individual ages"""
    if pd.isna(age_str) or age_str == 'N/A':
        return []
    
    age_str = str(age_str).lower()
    ages = []
    
    # Pattern: "2 - 5 years" or "2-5 years" or "2 to 5 years"
    match = re.search(r'(\d+)\s*[-–—to]\s*(\d+)\s*years?', age_str)
    if match:
        min_age = int(match.group(1))
        max_age = int(match.group(2))
        # Expand range: if 4-8, include 4, 5, 6, 7, 8
        ages = list(range(min_age, max_age + 1))
        return ages
    
    # Pattern: "Baby - 2 years" or "0-2 years"
    match = re.search(r'(?:baby|0)\s*[-–—to]\s*(\d+)\s*years?', age_str)
    if match:
        max_age = int(match.group(1))
        ages = list(range(0, max_age + 1))
        return ages
    
    # Pattern: "12+ years" or "12 and up"
    match = re.search(r'(\d+)\s*[+]|(\d+)\s*and\s*up', age_str)
    if match:
        min_age = int(match.group(1) or match.group(2))
        # For 12+, we'll include ages 12-18
        ages = list(range(min_age, 19))
        return ages
    
    # Pattern: single age "5 years"
    match = re.search(r'^(\d+)\s*years?', age_str)
    if match:
        age = int(match.group(1))
        return [age]
    
    return []

# Parse ages and expand ranges
if not combined_df.empty:
    combined_df['age_list'] = combined_df['Reading Age'].apply(parse_age_range)
    
    # Count how many books have valid ages
    books_with_ages = combined_df['age_list'].apply(lambda x: len(x) > 0).sum()
    print(f"Books with valid age information: {books_with_ages} out of {len(combined_df)}")
    
    # Show sample age expansions
    print("\nSample age range expansions:")
    sample_ages = combined_df[combined_df['age_list'].apply(lambda x: len(x) > 0)].head(5)
    for idx, row in sample_ages.iterrows():
        print(f"  '{row['Reading Age']}' -> {row['age_list']}")


## 4. Create Expanded Dataset (One Row per Age-Topic Combination)


In [ ]:
# Expand dataset: for each book, create a row for each age in its age range and each topic
expanded_rows = []

if not combined_df.empty:
    for idx, row in combined_df.iterrows():
        ages = row['age_list']
        topics = row['topics']
        
        if ages and topics:
            # Create a row for each age-topic combination
            for age in ages:
                for topic in topics:
                    expanded_rows.append({
                        'book_title': row.get('Book Title', ''),
                        'age': age,
                        'topic': topic,
                        'reading_age': row.get('Reading Age', ''),
                        'grade_level': row.get('Grade Level', ''),
                        'source_file': row.get('source_file', '')
                    })
        elif ages:
            # If no topics, still create rows for ages
            for age in ages:
                expanded_rows.append({
                    'book_title': row.get('Book Title', ''),
                    'age': age,
                    'topic': 'No Topic',
                    'reading_age': row.get('Reading Age', ''),
                    'grade_level': row.get('Grade Level', ''),
                    'source_file': row.get('source_file', '')
                })

expanded_df = pd.DataFrame(expanded_rows)

if not expanded_df.empty:
    print(f"Expanded dataset shape: {expanded_df.shape}")
    print(f"\nAge distribution:")
    print(expanded_df['age'].value_counts().sort_index().head(20))
    print(f"\nTop topics:")
    print(expanded_df['topic'].value_counts().head(20))


 t

In [ ]:
if not expanded_df.empty:
    # Get top 15 topics
    top_topics = expanded_df['topic'].value_counts().head(15).index.tolist()
    
    # Filter to top topics
    filtered_df = expanded_df[expanded_df['topic'].isin(top_topics)]
    
    # Create pivot table: age vs topic
    pivot_data = filtered_df.groupby(['age', 'topic']).size().reset_index(name='count')
    pivot_table = pivot_data.pivot(index='age', columns='topic', values='count').fillna(0)
    
    # Create heatmap
    plt.figure(figsize=(16, 10))
    sns.heatmap(pivot_table, annot=True, fmt='.0f', cmap='YlOrRd', cbar_kws={'label': 'Number of Books'})
    plt.title('Distribution of Topics Across Age Groups\n(If age is a range, topic appears in all ages in that range)', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Topic', fontsize=12, fontweight='bold')
    plt.ylabel('Age', fontsize=12, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    print(f"\nHeatmap shows {len(pivot_table)} age groups and {len(pivot_table.columns)} topics")


### 5.2 Distribution of Topics per Age (Stacked Bar Chart)


In [ ]:
if not expanded_df.empty:
    # Get top 10 topics
    top_topics = expanded_df['topic'].value_counts().head(10).index.tolist()
    
    # Filter to top topics and ages 0-18
    filtered_df = expanded_df[
        (expanded_df['topic'].isin(top_topics)) & 
        (expanded_df['age'] >= 0) & 
        (expanded_df['age'] <= 18)
    ]
    
    # Create pivot table
    pivot_data = filtered_df.groupby(['age', 'topic']).size().reset_index(name='count')
    pivot_table = pivot_data.pivot(index='age', columns='topic', values='count').fillna(0)
    
    # Create stacked bar chart
    fig, ax = plt.subplots(figsize=(16, 8))
    pivot_table.plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
    
    plt.title('Distribution of Topics per Age\n(Stacked Bar Chart)', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Age', fontsize=12, fontweight='bold')
    plt.ylabel('Number of Books', fontsize=12, fontweight='bold')
    plt.legend(title='Topic', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    print(f"\nChart shows distribution of top {len(top_topics)} topics across ages 0-18")


### 5.3 Topic Distribution per Age (Line Chart)


In [ ]:
if not expanded_df.empty:
    # Get top 10 topics
    top_topics = expanded_df['topic'].value_counts().head(10).index.tolist()
    
    # Filter to top topics and ages 0-18
    filtered_df = expanded_df[
        (expanded_df['topic'].isin(top_topics)) & 
        (expanded_df['age'] >= 0) & 
        (expanded_df['age'] <= 18)
    ]
    
    # Create pivot table
    pivot_data = filtered_df.groupby(['age', 'topic']).size().reset_index(name='count')
    pivot_table = pivot_data.pivot(index='age', columns='topic', values='count').fillna(0)
    
    # Create line chart
    plt.figure(figsize=(16, 8))
    for topic in top_topics:
        if topic in pivot_table.columns:
            plt.plot(pivot_table.index, pivot_table[topic], marker='o', label=topic, linewidth=2, markersize=6)
    
    plt.title('Topic Distribution per Age\n(Line Chart - Top 10 Topics)', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Age', fontsize=12, fontweight='bold')
    plt.ylabel('Number of Books', fontsize=12, fontweight='bold')
    plt.legend(title='Topic', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.xticks(range(0, 19))
    plt.tight_layout()
    plt.show()
    
    print(f"\nLine chart shows how top {len(top_topics)} topics vary across ages 0-18")


### 5.4 Age Distribution Overall


In [ ]:
if not expanded_df.empty:
    # Filter ages 0-18
    age_dist = expanded_df[(expanded_df['age'] >= 0) & (expanded_df['age'] <= 18)]['age'].value_counts().sort_index()
    
    plt.figure(figsize=(14, 6))
    age_dist.plot(kind='bar', color='steelblue', edgecolor='black')
    
    plt.title('Overall Age Distribution\n(Number of Books per Age)', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Age', fontsize=12, fontweight='bold')
    plt.ylabel('Number of Books', fontsize=12, fontweight='bold')
    plt.xticks(rotation=0)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\nTotal book-age combinations: {age_dist.sum()}")
    print(f"Ages with most books: {age_dist.head(10).to_dict()}")


### 5.5 Topic Distribution Overall


In [ ]:
if not expanded_df.empty:
    # Get top 20 topics
    topic_dist = expanded_df['topic'].value_counts().head(20)
    
    plt.figure(figsize=(14, 8))
    topic_dist.plot(kind='barh', color='coral', edgecolor='black')
    
    plt.title('Overall Topic Distribution\n(Top 20 Topics)', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Number of Books', fontsize=12, fontweight='bold')
    plt.ylabel('Topic', fontsize=12, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\nTop 10 topics:")
    for topic, count in topic_dist.head(10).items():
        print(f"  {topic}: {count}")


### 5.6 Overall Topic Distribution (From Original Books, Not Expanded)


In [ ]:
# Plot topics from original books (not expanded by age)
if not combined_df.empty:
    # Get all topics from original books
    all_topics_original = []
    for topics_list in combined_df['topics']:
        all_topics_original.extend(topics_list)
    
    topic_counts_original = Counter(all_topics_original)
    top_topics_original = pd.Series(dict(topic_counts_original.most_common(20)))
    
    plt.figure(figsize=(14, 8))
    top_topics_original.plot(kind='barh', color='teal', edgecolor='black')
    
    plt.title('Topic Distribution (From Original Books)\n(Top 20 Topics - No Age Information)', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Number of Books', fontsize=12, fontweight='bold')
    plt.ylabel('Topic', fontsize=12, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\nTotal unique topics: {len(topic_counts_original)}")
    print(f"\nTop 10 topics (from original books):")
    for topic, count in top_topics_original.head(10).items():
        print(f"  {topic}: {count}")
